# Playing with a Real LLM (HF - HuggingFace)

In earlier notebooks we built the *ideas*:

- tokenization
- embeddings
- attention
- decoder + sampling
- KV cache

Here we run a **real small LLM from Hugging Face** (https://huggingface.co/) and see those ideas in action.

## 1. Setup and Model

If you already have `transformers` and `torch` locally, you can skip it.

### If IDE is ready, you can skip this.
```bash
!pip install -q transformers torch accelerate
```

In [ ]:
import time
from typing import Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

# Fix randomness so results are repeatable
set_seed(42)

# Pick device: use GPU if available, otherwise CPU
if torch.backends.mps.is_available():
    device = "mps" # On M1/M2/M3 Macs, PyTorch uses Apple’s Metal backend (MPS) to access the GPU.
elif torch.cuda.is_available():
    device = "cuda" # For NVIDIA GPUs
else:
    device = "cpu"
print("Using device:", device)

# Use a small GPT-style model so it runs almost anywhere (distilgpt2 has 82M parameters)
model_name = "distilgpt2"  # you can later try bigger models if you have more RAM/GPU

# Load tokenizer (text <-> token IDs) and model (IDs -> logits)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

print("Loaded model:", model_name)

## 2. Tokenization and Logits (One Forward Pass)

We’ll:

1. Tokenize a short sentence.  
2. Run it through the model.  
3. Look at the **next‑token probabilities** at the last position.


In [ ]:
# A short input sentence
text = "Large language models are"

# Tokenize: this turns text into token IDs + attention mask
enc = tokenizer(text, return_tensors="pt").to(device)

print("Input IDs shape :", enc["input_ids"].shape)   # [batch, seq_len]
print("Attention mask :", enc["attention_mask"].shape)

# Show token IDs and the corresponding tokens
input_ids = enc["input_ids"][0]
print("\nToken IDs:", input_ids.tolist())
print("Tokens:")
print(tokenizer.convert_ids_to_tokens(input_ids))

In [ ]:
# Forward pass: get logits (scores for each vocab token)
with torch.no_grad():
    out = model(**enc)

logits = out.logits  # shape: [batch, seq_len, vocab_size]
print("Logits shape:", logits.shape)

# Take the logits at the last position (next-token distribution)
last_logits = logits[0, -1, :]                # [vocab_size]
probs = torch.softmax(last_logits, dim=-1)    # convert scores -> probabilities

# Show top 10 most likely next tokens
topk = torch.topk(probs, k=10)
top_probs = topk.values
top_indices = topk.indices
top_tokens = tokenizer.convert_ids_to_tokens(top_indices.tolist())

print("\nTop 10 next-token candidates:")
for tok, p in zip(top_tokens, top_probs.tolist()):
    print(f"{tok:>10s}  ->  {p:.4f}")

## 3. Generation and Sampling (Temperature / Top‑k / Top‑p)

Now we’ll use `model.generate()` to see how **sampling settings** change the output:

- `temperature=0.0` → greedy, deterministic  
- higher temperature → more randomness  
- `top_k` / `top_p` → restrict which tokens we sample from


In [ ]:
def generate_with_settings(
    prompt: str,
    max_new_tokens: int = 10,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
    do_sample: bool = True,
):
    """Helper to print generations with different sampling settings.

    This function:
    1. Tokenizes the prompt.
    2. Calls model.generate() with the given decoding options.
    3. Decodes and prints the full output text.
    """
    print("=" * 80)
    print(f"Prompt      : {prompt!r}")
    print(f"temperature : {temperature}")
    print(f"top_k       : {top_k}")
    print(f"top_p       : {top_p}")
    print("=" * 80)

    enc = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode all tokens (prompt + generated) back into text
    generated_text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(generated_text)
    print()

In [ ]:
prompt = "Large language models will"

# 1) Greedy decoding: always pick the most likely next token
generate_with_settings(
    prompt,
    temperature=0.0,     # temperature=0 => argmax / greedy
    top_k=None,
    top_p=None,
    do_sample=False,
)

# 2) Slight randomness: small temperature
generate_with_settings(
    prompt,
    temperature=0.7,
    top_k=None,
    top_p=None,
    do_sample=True,
)

# 3) Temperature + top-k + top-p: more controlled creativity
generate_with_settings(
    prompt,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    do_sample=True,
)


## 4. KV Cache Speedup (Timing Only)

Internally, each attention layer keeps **Key (K)** and **Value (V)** vectors for past tokens.

- Without cache: every new token recomputes attention over the whole prefix.  
- With cache: we reuse old K/V and only compute for the **new token**.

Hugging Face exposes this via `use_cache=True/False`. We’ll just compare timings.


In [ ]:
def timed_generate(prompt: str, max_new_tokens: int = 80, use_cache: bool = True) -> float:
    """Generate text and return time taken.

    We use greedy decoding so timing is less noisy.
    The actual content of the output is not important here.
    """
    enc = tokenizer(prompt, return_tensors="pt").to(device)

    start = time.perf_counter()
    with torch.no_grad():
        _ = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            use_cache=use_cache,
            pad_token_id=tokenizer.eos_token_id,
        )
    end = time.perf_counter()

    return end - start


prompt = "In the future, artificial intelligence systems will"

t_no_cache = timed_generate(prompt, max_new_tokens=800, use_cache=False)
t_cache    = timed_generate(prompt, max_new_tokens=800, use_cache=True)

print(f"Time without KV cache : {t_no_cache:.4f} seconds")
print(f"Time with KV cache    : {t_cache:.4f} seconds")

# Note: On small models and short sequences the difference may be small,
#       but on big models + long prompts KV cache is a huge speed boost.